# Spotify Audio Embedding Exploration

This notebook deliberately goes beyond the descriptive plots in `SpotifyEDA.ipynb`. It treats eight audio attributes as a compact **track embedding**, then studies geometry, PCA projections, clusters, and nearest-neighbor recommendations.

In [1]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

Matplotlib is building the font cache; this may take a moment.


In [2]:
DATA_PATH = Path('../data/spotify_tracks.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/spotify_tracks.csv')

df = pd.read_csv(DATA_PATH).drop(columns=['Unnamed: 0'], errors='ignore')
df = df.dropna(subset=['track_id', 'track_genre']).drop_duplicates('track_id').reset_index(drop=True)
print(f'{len(df):,} unique tracks across {df.track_genre.nunique()} genres')
df[['track_name', 'artists', 'track_genre']].head()

5,494 unique tracks across 7 genres


,track_name,artists,track_genre
0,Comedy,Gen Hoshino,acoustic
1,Ghost - Acoustic,Ben Woodward,acoustic
2,To Begin Again,Ingrid Michaelson;ZAYN,acoustic
3,Can't Help Falling In Love,Kina Grannis,acoustic
4,Hold On,Chord Overstreet,acoustic


## 1. Build the embedding

The embedding excludes popularity because popularity is an outcome/context variable rather than an acoustic property. Tempo is standardized together with the bounded Spotify features so that its larger numerical scale does not dominate distance.

In [3]:
FEATURES = [
    'danceability', 'energy', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo'
]

X = df[FEATURES].astype(float)
scaler = StandardScaler()
embedding = scaler.fit_transform(X)

pd.DataFrame({
    'feature': FEATURES,
    'mean': scaler.mean_,
    'scale': scaler.scale_
}).round(3)

,feature,mean,scale
0,danceability,0.533,0.173
1,energy,0.575,0.289
2,speechiness,0.065,0.064
3,acousticness,0.378,0.368
4,instrumentalness,0.242,0.372
5,liveness,0.178,0.151
6,valence,0.447,0.268
7,tempo,119.606,30.604


## 2. Project the embedding with PCA

PCA gives a global two-dimensional view. The loading table below is essential: it tells us which audio attributes shape each axis instead of treating the projection as a mysterious picture.

In [4]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(embedding)
df['pc1'], df['pc2'] = coords[:, 0], coords[:, 1]

loadings = pd.DataFrame(
    pca.components_.T,
    index=FEATURES,
    columns=['PC1', 'PC2']
)
print('Explained variance:', np.round(pca.explained_variance_ratio_, 3),
      '| total =', round(pca.explained_variance_ratio_.sum(), 3))
loadings.style.background_gradient(cmap='vlag', axis=None).format('{:.3f}')

Explained variance: [0.375 0.143] | total = 0.519


,PC1,PC2
danceability,0.332,0.621
energy,0.501,-0.208
speechiness,0.222,-0.063
acousticness,-0.455,0.246
instrumentalness,-0.373,-0.042
liveness,0.175,-0.385
valence,0.432,0.352
tempo,0.160,-0.481


In [5]:
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df, x='pc1', y='pc2', hue='track_genre',
    size='popularity', sizes=(12, 90), alpha=.55, linewidth=0
)
plt.title('Audio-feature embedding projected to two principal components')
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 3. Let the geometry suggest clusters

We compare several values of *k* using silhouette score, then inspect the winning segmentation. These are acoustic profiles—not genre labels—and overlap between genres is expected.

In [6]:
scores = {}
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE).fit_predict(embedding)
    scores[k] = silhouette_score(embedding, labels, sample_size=min(3000, len(df)), random_state=RANDOM_STATE)

best_k = max(scores, key=scores.get)
pd.Series(scores, name='silhouette_score').rename_axis('k').to_frame().round(3)

,silhouette_score
k,
2,0.272
3,0.272
4,0.211
5,0.205
6,0.221
7,0.212
8,0.213


In [7]:
model = KMeans(n_clusters=best_k, n_init=30, random_state=RANDOM_STATE)
df['cluster'] = model.fit_predict(embedding)

profiles = df.groupby('cluster')[FEATURES + ['popularity']].mean()
profile_z = (profiles[FEATURES] - X.mean()) / X.std()
plt.figure(figsize=(12, max(3, best_k * .7)))
sns.heatmap(profile_z, cmap='vlag', center=0, annot=True, fmt='.1f')
plt.title(f'Standardized acoustic profile of {best_k} embedding clusters')
plt.tight_layout()
plt.show()

pd.crosstab(df.cluster, df.track_genre, normalize='index').round(2)

track_genre,acoustic,afrobeat,alt-rock,alternative,ambient,anime,black-metal
cluster,,,,,,,
0,0.29,0.03,0.03,0.02,0.48,0.14,0.01
1,0.13,0.26,0.26,0.10,0.03,0.20,0.02


## 4. Use the embedding for track similarity

Cosine distance compares the direction of standardized feature profiles. The function makes the notebook interactive without requiring a Spotify account or external API.

In [8]:
neighbors = NearestNeighbors(metric='cosine').fit(embedding)

def similar_tracks(query, n=8):
    mask = (df.track_name.str.contains(query, case=False, na=False) |
            df.artists.str.contains(query, case=False, na=False))
    if not mask.any():
        raise ValueError(f'No track or artist matched {query!r}')
    source_idx = df.index[mask][0]
    distances, indices = neighbors.kneighbors(embedding[[source_idx]], n_neighbors=n + 1)
    result = df.loc[indices[0][1:], ['track_name', 'artists', 'track_genre', 'popularity']].copy()
    result.insert(0, 'similarity', 1 - distances[0][1:])
    print('Query:', df.loc[source_idx, 'track_name'], '—', df.loc[source_idx, 'artists'])
    return result.reset_index(drop=True).style.format({'similarity': '{:.1%}', 'popularity': '{:.0f}'})

similar_tracks('Comedy')

Query: Comedy — Gen Hoshino


,similarity,track_name,artists,track_genre,popularity
0,95.6%,Look For The Good (Single Version),Jason Mraz,acoustic,21
1,92.8%,Pop Virus,Gen Hoshino,acoustic,47
2,89.8%,"Hard Talk, Pt. 2",TJ Cream;Marko,afrobeat,21
3,88.0%,Mr. P-Mosh,Plastilina Mosh,afrobeat,23
4,87.2%,Make Me Feel,Janelle Monáe,alternative,0
5,87.2%,Make Me Feel,Janelle Monáe,alternative,0
6,87.2%,Make Me Feel,Janelle Monáe,alternative,0
7,86.1%,Authentic,A Mose;TBabz,afrobeat,20


## Takeaways and next experiments

- PCA reveals the strongest continuous directions in the audio-feature space, while its loadings make those directions interpretable.
- Clusters summarize recurring acoustic profiles; they should not be read as ground-truth genres.
- Nearest neighbors turn the same representation into a small content-based recommender.
- A useful next step would compare this standardized baseline with UMAP or learned embeddings, using held-out playlist co-occurrence as an evaluation signal.